Improve SVD RMSE

SECTION 1 Setup

(Import libraries)

In [ ]:
import pandas as pd

train_df = pd.read_csv(
    "train_per_user_temporal.csv"
)

test_df = pd.read_csv(
    "test_per_user_temporal.csv"
)

print(train_df.shape)
print(test_df.shape)

(1506935, 4)
(493065, 4)


In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import mean_squared_error

!pip install scikit-surprise -q
from surprise import Dataset
from surprise import Reader
from surprise import SVD

SECTION 2 RMSE Function

(Evaluation metric)

In [ ]:
def rmse(actual, predicted):
    return np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

SECTION 3 Build Surprise Dataset

(Prepare training data)

In [ ]:
reader = Reader(
    rating_scale=(1,5)
)

surprise_data = Dataset.load_from_df(
    train_df[
        [
            "user_id",
            "movie_id",
            "rating"
        ]
    ],
    reader
)

trainset = (
    surprise_data
    .build_full_trainset()
)

SECTION 4 Candidate Models

(Different SVD configurations)

In [ ]:
configs = [

    {
        "name":"SVD_50F_20E",
        "n_factors":50,
        "n_epochs":20,
        "lr_all":0.005,
        "reg_all":0.02
    },

    {
        "name":"SVD_100F_30E",
        "n_factors":100,
        "n_epochs":30,
        "lr_all":0.005,
        "reg_all":0.02
    },

    {
        "name":"SVD_100F_50E",
        "n_factors":100,
        "n_epochs":50,
        "lr_all":0.003,
        "reg_all":0.05
    }

]

SECTION 5 Train & Evaluate

(Find best SVD configuration)

In [ ]:
results = []

In [ ]:
for cfg in configs:

    print(
        f"\nTraining {cfg['name']}"
    )

    model = SVD(
        n_factors=cfg["n_factors"],
        n_epochs=cfg["n_epochs"],
        lr_all=cfg["lr_all"],
        reg_all=cfg["reg_all"],
        random_state=42
    )

    model.fit(trainset)

    predictions = []

    for row in tqdm(
        test_df.itertuples(),
        total=len(test_df),
        leave=False
    ):

        pred = model.predict(
            row.user_id,
            row.movie_id
        )

        predictions.append(
            pred.est
        )

    score = rmse(
        test_df["rating"],
        predictions
    )

    results.append([
        cfg["name"],
        score
    ])

    print(
        "RMSE:",
        round(score,4)
    )


Training SVD_50F_20E


RMSE: 0.9735

Training SVD_100F_30E


RMSE: 0.9921

Training SVD_100F_50E


RMSE: 0.9703


SECTION 6 Results Table

(Compare all configurations)

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "RMSE"
    ]
)

results_df.sort_values(
    "RMSE"
)

,Model,RMSE
2,SVD_100F_50E,0.970288
0,SVD_50F_20E,0.973522
1,SVD_100F_30E,0.992100


SECTION 7 Best Model

(Identify strongest configuration)

In [ ]:
best_model = results_df.sort_values(
    "RMSE"
).iloc[0]

best_model

,2
Model,SVD_100F_50E
RMSE,0.970288


SECTION 8 RMSE Comparison Plot

(Visual comparison)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(
    figsize=(10,5)
)

plt.bar(
    results_df["Model"],
    results_df["RMSE"]
)

plt.title(
    "SVD Hyperparameter Tuning"
)

plt.ylabel(
    "RMSE"
)

plt.xticks(
    rotation=45
)

plt.show()